|<h2>Course:</h2>|<h1><a href="https://derivingsystems.com/course.html" target="_blank">Build your own vLLM: inference engines from the memory system up</a></h1>|
|-|:-:|
|<h2>Part 1:</h2>|<h1>The Naive Loop<h1>|
|<h2>Section:</h2>|<h1>The KV cache<h1>|
|<h2>Lecture:</h2>|<h1><b>CodeChallenge HELPER: write both loops and make them agree<b></h1>|

<br>

<h5><b>Course repo:</b> <a href="https://github.com/Venugopalan2610/vllm-from-scratch" target="_blank">github.com/Venugopalan2610/vllm-from-scratch</a></h5>
<h5><b>The derivations:</b> <a href="https://derivingsystems.com" target="_blank">derivingsystems.com</a></h5>
<i>The notebooks build the intuition. The ladder in app/ makes you build the thing.</i>

In [ ]:
# The naive loop runs a forward pass at every length from 1 to N. So it asks
# the allocator for N different block sizes. The default allocator keeps all
# of them, and then it runs out of room on a card with 12 GB free. This mode
# grows one segment instead. Set it BEFORE you import torch.
import os
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

# find the repo root. The directory you start from does not matter.
import sys
from pathlib import Path
ROOT = next(folder for folder in [Path.cwd(), *Path.cwd().parents]
            if (folder/'cudalib').is_dir())
sys.path.insert(0, str(ROOT))

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

import cudalib

You must write two generation loops. Both must produce the same text.

One loop recomputes the whole prefix at every step. The other keeps K and V
and feeds the model one token at a time. Write both. Prove that they agree.
Then find what the cache bought you.

This is a small version of stage 02. It uses HuggingFace's cache instead of
one that you built.

In [ ]:
### run this cell

MODEL = 'Qwen/Qwen3-0.6B'
tokenizer = AutoTokenizer.from_pretrained(MODEL)

def load(dtype):
  return AutoModelForCausalLM.from_pretrained(MODEL, dtype=dtype).cuda().eval()

model = load(torch.bfloat16)
prompt = tokenizer('The capital of France is', return_tensors='pt').input_ids.cuda()
print(tokenizer.decode(prompt[0]))

# Exercise 1: the naive loop

Use greedy decoding and no cache. The whole sequence passes through the model
at every step.

In [ ]:
@torch.inference_mode()
def naive_generate(model, prompt_ids, num_tokens):
  token_ids = prompt_ids.clone()
  for _ in range(num_tokens):

    # the whole prefix, every time. no cache.
    logits = 

    # greedy: the most likely next token
    next_token = 

    # append it and go round again
    token_ids = 
  return token_ids

result = naive_generate(model, prompt, 20)
print(tokenizer.decode(result[0]))

# Exercise 2: the cached loop

The prompt passes through once. After that, each step sees one token. The
model reads the rest from `past_key_values`.

The trap is the input to the second pass. Feed the whole sequence again and
the cache silently appends it to a prefix that it already holds.

In [ ]:
@torch.inference_mode()
def cached_generate(model, prompt_ids, num_tokens):

  # the prompt goes through once, in full. This is prefill.
  result = 
  cache = 
  next_token = 
  token_ids = torch.cat([prompt_ids, next_token], dim=1)

  # after that, one token at a time. This is decode.
  # careful: what exactly do you feed the model on this pass?
  for _ in range(num_tokens-1):
    result = 
    cache = 
    next_token = 
    token_ids = torch.cat([token_ids, next_token], dim=1)
  return token_ids

result = cached_generate(model, prompt, 20)
print(tokenizer.decode(result[0]))

# Exercise 3: do they agree?

You use the same model, the same prompt, and the same greedy rule. The two
loops must write the same sentence.

In [ ]:
naive_tokens = naive_generate(model, prompt, 24)
cached_tokens = cached_generate(model, prompt, 24)

print('identical:', torch.equal(naive_tokens, cached_tokens))
print('\nnaive :', tokenizer.decode(naive_tokens[0]))
print('cached:', tokenizer.decode(cached_tokens[0]))

# if they differ, where?
first_difference = 
print(f'\nfirst token that differs: {first_difference}')

# Exercise 4: what did the cache buy?

Time both loops at several lengths. One loop is linear in the number of
tokens. The other is not. So the gap must grow as you make more tokens.

The first cell of this notebook sets an allocator option. That option is not
superstition. The naive loop runs a forward pass at 512 different sequence
lengths, so it asks the allocator for 512 different block sizes. The default
allocator keeps all of them. Without the option this notebook reports an
out-of-memory warning on a card with 12 GB free.

Every new shape has a cost. You meet that fact twice more. On the JAX track a
new shape causes a recompile. In stage 12 it is the reason to capture CUDA
graphs at bucketed batch sizes.

In [ ]:
import gc

for num_tokens in (32, 128, 512):
  gc.collect()
  torch.cuda.empty_cache()
  ms_naive = 
  ms_cached = 
  print(f'{num_tokens:>4} tokens: naive {ms_naive/1000:6.2f} s   '
        f'cached {ms_cached/1000:5.2f} s   {ms_naive/ms_cached:5.2f}x')

# Exercise 5: now repeat Exercise 3 in fp32

You use the same two loops and the same prompt. One thing changes. This
exercise runs last, because fp32 weights are twice the size and crowd the
card.

In [ ]:
# load the same model in float32 and run both loops again
exact = 

naive_tokens = 
cached_tokens = 
print('fp32 identical:', torch.equal(naive_tokens, cached_tokens))

# fp32 is twice the weights. Give the memory back before the timings,
# or the next cell fragments the allocator and thrashes.
import gc
del exact, naive_tokens, cached_tokens
gc.collect()
torch.cuda.empty_cache()

### Before you open the solution

Two of your results look wrong. Write an explanation for each one before you
read mine.

1. In bf16 the two loops produce different sentences. In fp32 they do not.
   Both loops are correct. So what differs in the arithmetic? And why does a
   very small change in a logit change a word?
2. The speedup is much smaller than "quadratic against linear" suggests, and
   at 32 tokens the cache can lose. Where does a decode step on a 0.6B model
   spend its time? Which later stage attacks that?

Then build this properly, with a cache that you own:

    ./vc guide 2